# Exploración de datos de Instacart con Python y Pandas

## Momento 1. Comprensión del problema y reconocimiento de los datos

### Situación problemática

El equipo de operaciones de Instacart dispone de información sobre pedidos, clientes, productos, pasillos y departamentos, pero necesita comprender qué contienen esos registros y cómo consultarlos para estudiar los hábitos de compra. Como aprendiz en el rol de analista de datos, exploraré los cinco archivos con Python y Pandas, identificaré sus estructuras y consultaré registros mediante selección y filtros. También calcularé indicadores básicos de pedidos y recompras. La exploración inicial permitirá detectar tipos de datos, valores ausentes y relaciones entre tablas antes de interpretar resultados.

**Pregunta orientadora:** ¿Cómo utilizar Python y Pandas para explorar los datos de Instacart, identificar las características de los pedidos y productos, y obtener información relevante sobre los hábitos de compra de sus clientes?

### Descripción de los archivos

Los conteos se obtuvieron localmente con Pandas, usando `sep=';'`, que es el separador indicado por la guía.

| Archivo | ¿Qué información guarda? | Cada fila representa… | Registros | Columnas |
|---|---|---|---:|---|
| `instacart_orders.csv` | Identificadores del pedido y del usuario, número de pedido, día y hora, y días desde el pedido anterior. | Un pedido realizado por un usuario. | 478.967 | 6: `order_id`, `user_id`, `order_number`, `order_dow`, `order_hour_of_day`, `days_since_prior_order` |
| `products.csv` | Catálogo de productos y su clasificación por pasillo y departamento. | Un producto del catálogo. | 49.694 | 4: `product_id`, `product_name`, `aisle_id`, `department_id` |
| `aisles.csv` | Identificadores y nombres de los pasillos. | Un pasillo del catálogo. | 134 | 2: `aisle_id`, `aisle` |
| `departments.csv` | Identificadores y nombres de los departamentos. | Un departamento del catálogo. | 21 | 2: `department_id`, `department` |
| `order_products.csv` | Productos asociados a pedidos, su posición al agregarse al carrito y si ya se habían pedido antes. | Un producto incluido en un pedido; un pedido puede aparecer en varias filas. | 4.545.007 | 4: `order_id`, `product_id`, `add_to_cart_order`, `reordered` |

`order_products.csv` contiene muchos más registros porque guarda una fila por producto de cada pedido. Como un pedido puede contener varios productos, su `order_id` se repite en distintas filas.

### Identificadores y relaciones entre tablas

- `instacart_orders.order_id` identifica un pedido; `instacart_orders.user_id` identifica al cliente que lo realizó.
- `products.product_id` identifica un producto del catálogo.
- En `order_products`, `order_id` y `product_id` vinculan cada artículo con su pedido y con el catálogo de productos.
- `products.aisle_id` se relaciona con `aisles.aisle_id`; `products.department_id` se relaciona con `departments.department_id`.

### Preguntas de análisis iniciales

1. ¿Cuántos pedidos se realizaron a las 10:00 y cuántos de ellos corresponden a clientes con más de 10 pedidos registrados?
2. ¿Cuántas filas de artículos corresponden a productos que ya se habían pedido antes (`reordered == 1`)?
3. ¿Cuál es el promedio de días transcurridos desde el pedido anterior (`days_since_prior_order`), considerando los valores disponibles?

### ¿Por qué explorar antes de analizar?

La exploración permite comprobar que cada archivo se leyó con las columnas esperadas, entender qué representa cada fila, conocer el tamaño y tipo de los datos, detectar valores ausentes y reconocer las relaciones entre tablas. Sin estas comprobaciones se podrían contar artículos como si fueran pedidos o usuarios, combinar información incorrectamente o interpretar promedios sin considerar los datos ausentes.

**Nota de importación:** los CSV usan punto y coma como separador; se cargarán con `pd.read_csv(..., sep=';')`.

---

## Momento 2. Importación y exploración inicial

Se cargarán los cinco CSV desde la carpeta `data/`. Para cada DataFrame se mostrarán los primeros y últimos registros; después se compararán sus dimensiones con `shape`.

## Momento 3. Estructura y calidad inicial

Se revisarán `columns`, `dtypes` e `info()`. Una tabla de diagnóstico resumirá filas, columnas y valores ausentes. En esta fase solo se identifican posibles problemas; no se eliminan ni corrigen registros.

## Momento 4. Indexación y filtrado lógico

Se resolverán los ocho filtros solicitados y dos filtros combinados. Para cada ejercicio se mostrará el criterio, el número de filas coincidentes y una muestra del resultado.

## Momento 5. Indicadores básicos

Se calcularán las cantidades solicitadas, promedios, suma de `reordered` y descripciones estadísticas. Los resultados se interpretarán distinguiendo filas de pedidos, pedidos y clientes.

## Momento 6. Síntesis y conclusiones

Se responderán las tres preguntas iniciales, se resumirán hallazgos y limitaciones, y se redactarán tres conclusiones sustentadas en los resultados.

**Lista de chequeo:** la guía identifica el instrumento de evaluación como “lista de chequeo”, pero no incluye sus criterios detallados. Se contrastará el notebook con la guía disponible y se incorporará la lista cuando se comparta.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

DATA_DIR = Path("data")
if not DATA_DIR.exists():
    DATA_DIR = Path("../data")

orders = pd.read_csv(DATA_DIR / "instacart_orders.csv", sep=";")
products = pd.read_csv(DATA_DIR / "products.csv", sep=";")
aisles = pd.read_csv(DATA_DIR / "aisles.csv", sep=";")
departments = pd.read_csv(DATA_DIR / "departments.csv", sep=";")
order_products = pd.read_csv(DATA_DIR / "order_products.csv", sep=";")

dataframes = {
    "orders": orders,
    "products": products,
    "aisles": aisles,
    "departments": departments,
    "order_products": order_products,
}

print("Archivos cargados:", ", ".join(dataframes))

Archivos cargados: orders, products, aisles, departments, order_products


In [2]:
for name, dataframe in dataframes.items():
    print(f"\n========== {name}: primeras 5 filas ==========")
    display(dataframe.head())
    print(f"========== {name}: últimas 5 filas ==========")
    display(dataframe.tail())

print("\n========== Dimensiones ==========")
for name, dataframe in dataframes.items():
    print(f"{name}: {dataframe.shape}")


========== orders: primeras 5 filas ==========


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,1515936,183418,11,6,13,30.0
1,1690866,163593,5,5,12,9.0
2,1454967,39980,4,5,19,2.0
3,1768857,82516,56,0,20,10.0
4,3007858,196724,2,4,12,17.0


========== orders: últimas 5 filas ==========


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
478962,3210681,5617,5,1,14,7.0
478963,3270802,112087,2,3,13,6.0
478964,885349,82944,16,2,11,6.0
478965,216274,4391,3,3,8,8.0
478966,2071924,1730,18,1,14,15.0



========== products: primeras 5 filas ==========


...


========== order_products: primeras 5 filas ==========


,order_id,product_id,add_to_cart_order,reordered
0,2141543,11440,17.0,0
1,567889,1560,1.0,1
2,2261212,26683,1.0,1
3,491251,8670,35.0,1
4,2571142,1940,5.0,1


========== order_products: últimas 5 filas ==========


,order_id,product_id,add_to_cart_order,reordered
4545002,577211,15290,12.0,1
4545003,1219554,21914,9.0,0
4545004,692640,47766,4.0,1
4545005,319435,691,8.0,1
4545006,1398151,28733,9.0,0



========== Dimensiones ==========
orders: (478967, 6)
products: (49694, 4)
aisles: (134, 2)
departments: (21, 2)
order_products: (4545007, 4)


### Respuestas del Momento 2

- **¿Cuál DataFrame contiene más registros?** `order_products`, con 4.545.007 filas. Cada fila corresponde a un producto dentro de un pedido, no a un pedido completo.
- **¿Cuál contiene menos columnas?** `aisles` y `departments`, con 2 columnas cada uno.
- **¿Qué diferencias hay entre pedidos y productos?** `orders` describe cada pedido y sus datos de fecha/hora y usuario; `products` describe los productos del catálogo y su clasificación. `order_products` registra qué productos se incluyeron en cada pedido.
- **¿Por qué revisar las dimensiones?** `shape` permite conocer el volumen y la estructura de cada tabla, comprobar que la carga y el separador fueron correctos, y evitar confundir filas de artículos con pedidos o clientes.

## Momento 3. Reconocimiento de la estructura y calidad inicial de los datos

Las siguientes cinco celdas revisan columnas, tipos e información no nula. `columns` y `dtypes` son atributos; `info()` es un método que imprime el resumen. `show_counts=True` pide a pandas mostrar el conteo no nulo incluso para la tabla grande `order_products`. En pandas 3, las columnas de texto se muestran como `str`; en versiones anteriores era habitual ver `object`.

Compara cada conteo no nulo con el total de filas. Si el conteo es menor, la diferencia corresponde a valores ausentes. En esta fase solo los identificamos: no se rellenan ni se eliminan.

In [3]:
print("orders.columns:")
print(orders.columns)
print("\norders.dtypes:")
print(orders.dtypes)
print("\norders.info():")
orders.info(show_counts=True)

orders.columns:
Index(['order_id', 'user_id', 'order_number', 'order_dow', 'order_hour_of_day',
       'days_since_prior_order'],
      dtype='str')

orders.dtypes:
order_id                    int64
user_id                     int64
order_number                int64
order_dow                   int64
order_hour_of_day           int64
days_since_prior_order    float64
dtype: object

orders.info():
<class 'pandas.DataFrame'>
RangeIndex: 478967 entries, 0 to 478966
Data columns (total 6 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   order_id                478967 non-null  int64  
 1   user_id                 478967 non-null  int64  
 2   order_number            478967 non-null  int64  
 3   order_dow               478967 non-null  int64  
 4   order_hour_of_day       478967 non-null  int64  
 5   days_since_prior_order  450148 non-null  float64
dtypes: float64(1), int64(5)
memory usage: 21.9 MB


In [4]:
print("products.columns:")
print(products.columns)
print("\nproducts.dtypes:")
print(products.dtypes)
print("\nproducts.info():")
products.info(show_counts=True)

products.columns:
Index(['product_id', 'product_name', 'aisle_id', 'department_id'], dtype='str')

products.dtypes:
product_id       int64
product_name       str
aisle_id         int64
department_id    int64
dtype: object

products.info():
<class 'pandas.DataFrame'>
RangeIndex: 49694 entries, 0 to 49693
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   product_id     49694 non-null  int64
 1   product_name   48436 non-null  str  
 2   aisle_id       49694 non-null  int64
 3   department_id  49694 non-null  int64
dtypes: int64(3), str(1)
memory usage: 1.5 MB


In [5]:
print("aisles.columns:")
print(aisles.columns)
print("\naisles.dtypes:")
print(aisles.dtypes)
print("\naisles.info():")
aisles.info(show_counts=True)

aisles.columns:
Index(['aisle_id', 'aisle'], dtype='str')

aisles.dtypes:
aisle_id    int64
aisle         str
dtype: object

aisles.info():
<class 'pandas.DataFrame'>
RangeIndex: 134 entries, 0 to 133
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   aisle_id  134 non-null    int64
 1   aisle     134 non-null    str  
dtypes: int64(1), str(1)
memory usage: 2.2 KB


In [6]:
print("departments.columns:")
print(departments.columns)
print("\ndepartments.dtypes:")
print(departments.dtypes)
print("\ndepartments.info():")
departments.info(show_counts=True)

departments.columns:
Index(['department_id', 'department'], dtype='str')

departments.dtypes:
department_id    int64
department         str
dtype: object

departments.info():
<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   department_id  21 non-null     int64
 1   department     21 non-null     str  
dtypes: int64(1), str(1)
memory usage: 468.0 bytes


In [7]:
print("order_products.columns:")
print(order_products.columns)
print("\norder_products.dtypes:")
print(order_products.dtypes)
print("\norder_products.info():")
order_products.info(show_counts=True)

order_products.columns:
Index(['order_id', 'product_id', 'add_to_cart_order', 'reordered'], dtype='str')

order_products.dtypes:
order_id               int64
product_id             int64
add_to_cart_order    float64
reordered              int64
dtype: object

order_products.info():
<class 'pandas.DataFrame'>
RangeIndex: 4545007 entries, 0 to 4545006
Data columns (total 4 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   order_id           4545007 non-null  int64  
 1   product_id         4545007 non-null  int64  
 2   add_to_cart_order  4544171 non-null  float64
 3   reordered          4545007 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 138.7 MB


### Hallazgos del Momento 3

#### Tipos de datos por columna

| Archivo | Columnas enteras (`int64`) | Columnas decimales (`float64`) | Columnas de texto (`str`) |
|---|---|---|---|
| `orders` | `order_id`, `user_id`, `order_number`, `order_dow`, `order_hour_of_day` | `days_since_prior_order` | — |
| `products` | `product_id`, `aisle_id`, `department_id` | — | `product_name` |
| `aisles` | `aisle_id` | — | `aisle` |
| `departments` | `department_id` | — | `department` |
| `order_products` | `order_id`, `product_id`, `reordered` | `add_to_cart_order` | — |

En pandas 3, los textos se muestran con el tipo `str`; en versiones anteriores era común ver `object`. Ambos indican valores de texto. Las columnas `days_since_prior_order` y `add_to_cart_order` son decimales porque contienen valores ausentes, representados por `NaN`.

#### Campos que relacionan los archivos

| Campo | Aparece en | Qué conecta |
|---|---|---|
| `order_id` | `orders`, `order_products` | Cada artículo con el pedido al que pertenece. |
| `product_id` | `products`, `order_products` | Cada artículo de un pedido con su producto del catálogo. |
| `aisle_id` | `products`, `aisles` | Cada producto con el nombre de su pasillo. |
| `department_id` | `products`, `departments` | Cada producto con el nombre de su departamento. |

`user_id` identifica al usuario en `orders`, pero estos cinco archivos no incluyen una tabla de usuarios para relacionarlo con otros datos de perfil.

#### Tabla de diagnóstico de valores ausentes

| Archivo | Filas | Columnas | Observaciones |
|---|---:|---:|---|
| `orders` | 478.967 | 6 | 28.819 valores ausentes en `days_since_prior_order`; las otras 5 columnas están completas. |
| `products` | 49.694 | 4 | 1.258 valores ausentes en `product_name`; las otras 3 columnas están completas. |
| `aisles` | 134 | 2 | No se observan valores ausentes. |
| `departments` | 21 | 2 | No se observan valores ausentes. |
| `order_products` | 4.545.007 | 4 | 836 valores ausentes en `add_to_cart_order`; las otras 3 columnas están completas. |

#### Respuestas a las preguntas de análisis

- **¿Qué diferencias existen entre `int64`, `float64` y `str`?** `int64` almacena enteros, `float64` almacena números decimales y `str` almacena texto. En estos datos, las columnas decimales incluyen valores ausentes.
- **¿Qué aporta `info()` que no da `shape`?** `shape` informa solo el número de filas y columnas. `info()` también muestra nombres de columnas, tipos, valores no nulos y memoria aproximada.
- **¿En qué columnas hay valores ausentes?** `orders.days_since_prior_order` (28.819), `products.product_name` (1.258) y `order_products.add_to_cart_order` (836).
- **¿Qué dificultades podrían causar?** Los promedios se calculan con valores disponibles y pueden no representar filas con dato faltante; los filtros numéricos no seleccionan los `NaN`; los nombres ausentes dificultan identificar algunos productos. Según la guía, aquí se registran los ausentes, pero no se corrigen ni eliminan.

#### Observación adicional sobre duplicados

En `orders`, 15 filas repiten un `order_id` ya presente. No se eliminaron: la actividad pide reconocer las características y limitaciones, no limpiar los datos en esta fase. Por ello, al interpretar conteos se distingue entre filas del DataFrame e identificadores de pedido únicos.

## Momento 4. Indexación y filtrado lógico

Cada ejercicio tiene una celda de código independiente y una explicación breve. Seleccionar columnas reduce las columnas visibles pero conserva las filas; filtrar selecciona filas que cumplen una condición y conserva las columnas. `loc` permite indicar filas y columnas de forma explícita.

In [8]:
ej1 = products.loc[:, ["product_id", "product_name"]]
print("Filas y columnas:", ej1.shape)
display(ej1.head())

Filas y columnas: (49694, 2)


,product_id,product_name
0,1,Chocolate Sandwich Cookies
1,2,All-Seasons Salt
2,3,Robust Golden Unsweetened Oolong Tea
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...
4,5,Green Chile Anytime Sauce


*Ejercicio 1.* Se seleccionaron `product_id` y `product_name`. El resultado conserva las 49.694 filas del catálogo y queda con 2 columnas (`(49.694, 2)`).

In [9]:
ej2 = products.loc[products["department_id"] == 1]
print("Registros:", ej2.shape[0], "de", products.shape[0])
display(ej2.head())

Registros: 4008 de 49694


,product_id,product_name,aisle_id,department_id
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
7,8,Cut Russet Potatoes Steam N' Mash,116,1
11,12,Chocolate Fudge Layer Cake,119,1
17,18,Pizza for One Suprema Frozen Pizza,79,1
29,30,Three Cheese Ziti- Marinara with Meatballs,38,1


*Ejercicio 2.* Se filtraron productos con `department_id == 1`. Quedaron 4.008 de 49.694 productos; el resultado incluye solo productos asignados al departamento con identificador 1.

In [10]:
ej3 = orders.loc[orders["order_hour_of_day"] == 10]
print("Registros:", ej3.shape[0], "de", orders.shape[0])
display(ej3.head())

Registros: 40578 de 478967


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
19,294269,107523,29,2,10,1.0
28,133707,182261,1,3,10,NaN
36,3378084,127629,21,1,10,20.0
49,2161799,87345,7,0,10,24.0
51,1739741,118405,48,2,10,3.0


*Ejercicio 3.* Se filtraron pedidos realizados a las 10:00. Quedaron 40.578 de 478.967 pedidos, aproximadamente el 8,5 % del archivo.

In [11]:
ej4 = orders.loc[orders["order_dow"] == 0]
print("Registros:", ej4.shape[0], "de", orders.shape[0])
display(ej4.head())

Registros: 84090 de 478967


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
3,1768857,82516,56,0,20,10.0
5,1843871,20084,5,0,11,15.0
7,2521283,97711,2,0,18,22.0
23,3406282,19356,21,0,0,14.0
25,1895419,169793,2,0,11,8.0


*Ejercicio 4.* Se filtraron pedidos con `order_dow == 0`. Quedaron 84.090 de 478.967. El archivo usa valores de día codificados; aquí se informa el código 0 sin asignarle un día específico porque no hay un diccionario de códigos en los CSV.

In [12]:
ej5 = orders.loc[orders["order_number"] > 10]
print("Registros:", ej5.shape[0], "de", orders.shape[0])
display(ej5.head())

Registros:

 244072 de 478967


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,1515936,183418,11,6,13,30.0
3,1768857,82516,56,0,20,10.0
6,2511640,148233,40,3,8,2.0
8,2266494,34954,40,5,15,4.0
10,1162566,43738,33,4,11,2.0


*Ejercicio 5.* Se filtraron pedidos cuyo número secuencial es mayor que 10. Quedaron 244.072 de 478.967 registros. El resultado cuenta pedidos de orden avanzado, no clientes únicos.

In [13]:
ej6 = orders.loc[orders["days_since_prior_order"] > 15]
print("Registros:", ej6.shape[0], "de", orders.shape[0])
display(ej6.head())

Registros: 110706 de 478967


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,1515936,183418,11,6,13,30.0
4,3007858,196724,2,4,12,17.0
7,2521283,97711,2,0,18,22.0
14,2843219,74699,4,6,11,30.0
26,1844247,122558,4,6,18,30.0


*Ejercicio 6.* Se filtraron pedidos con más de 15 días desde el pedido anterior. Quedaron 110.706 de 478.967 filas; los valores `NaN` no cumplen la comparación y no se incluyen. El promedio y otros cálculos de esta columna usan solo sus 450.148 valores disponibles.

In [14]:
ej7 = order_products.loc[order_products["reordered"] == 1]
print("Registros:", ej7.shape[0], "de", order_products.shape[0])
display(ej7.head())

Registros:

 2683838 de 4545007


,order_id,product_id,add_to_cart_order,reordered
1,567889,1560,1.0,1
2,2261212,26683,1.0,1
3,491251,8670,35.0,1
4,2571142,1940,5.0,1
5,2456893,21616,4.0,1


*Ejercicio 7.* Se filtraron artículos con `reordered == 1`. Quedaron 2.683.838 de 4.545.007 filas. Cada fila es una asociación producto-pedido; el conteo no equivale a productos únicos ni a clientes.

In [15]:
ej8 = order_products.loc[order_products["add_to_cart_order"] == 1]
print("Registros:", ej8.shape[0], "de", order_products.shape[0])
display(ej8.head())

Registros: 450046 de 4545007


,order_id,product_id,add_to_cart_order,reordered
1,567889,1560,1.0,1
2,2261212,26683,1.0,1
14,1961225,37553,1.0,1
16,639939,10017,1.0,1
23,750040,8518,1.0,0


*Ejercicio 8.* Se seleccionaron 450.046 artículos que ocupan la primera posición del carrito. Coincide con los 450.046 pedidos representados en `order_products`, pero hay 28.906 pedidos del archivo `orders` que no aparecen en `order_products`; por eso no se debe esperar que el conteo sea igual a todas las filas de `orders`.

### Filtros combinados

El operador `&` exige que ambas condiciones sean verdaderas. Cada condición debe ir entre paréntesis.

In [16]:
pedidos_combinados = orders.loc[
    (orders["order_hour_of_day"] == 10)
    & (orders["order_number"] > 10)
]
print("Registros:", pedidos_combinados.shape[0], "de", orders.shape[0])
display(pedidos_combinados.head())

Registros: 21537 de 478967


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
19,294269,107523,29,2,10,1.0
36,3378084,127629,21,1,10,20.0
51,1739741,118405,48,2,10,3.0
92,60724,110808,78,0,10,2.0
140,3156095,113920,14,2,10,2.0


Se aplicaron simultáneamente hora igual a 10 y número de pedido mayor que 10. Coinciden 21.537 pedidos de los 478.967 registros.

In [17]:
productos_combinados = products.loc[
    (products["department_id"] == 1)
    & (products["aisle_id"] > 50)
]
print("Registros:", productos_combinados.shape[0], "de", products.shape[0])
display(productos_combinados.head())

Registros: 1618 de 49694


,product_id,product_name,aisle_id,department_id
7,8,Cut Russet Potatoes Steam N' Mash,116,1
11,12,Chocolate Fudge Layer Cake,119,1
17,18,Pizza for One Suprema Frozen Pizza,79,1
45,46,Raisin Cinnamon Bagels 5 count,58,1
127,128,Organic Wild Blueberries,116,1


Se aplicaron simultáneamente `department_id == 1` y `aisle_id > 50`. Coinciden 1.618 productos de los 49.694 del catálogo. La comprobación confirmó que todos cumplen ambas condiciones.

### Respuestas a las preguntas del Momento 4

- **Seleccionar columnas y filtrar filas:** seleccionar columnas decide qué variables se muestran y conserva todas las filas; filtrar filas conserva las columnas y reduce los registros a los que cumplen una condición.
- **Ventaja de `loc[]`:** permite indicar explícitamente qué filas y columnas se consultan, lo que hace la selección más legible y precisa.
- **Varias condiciones simultáneas:** normalmente dejan igual o reducen la cantidad de filas, porque deben cumplirse todas las condiciones unidas con `&`.
- **Operadores lógicos:** `==`, `>`, `<`, `>=` y `!=` expresan criterios distintos. Usar el operador correcto y paréntesis en cada condición evita seleccionar registros incorrectos.

## Momento 5. Obtención e interpretación de indicadores básicos

La siguiente celda calcula los indicadores pedidos y muestra las estadísticas descriptivas. `count()` cuenta valores no nulos; `sum()` suma los valores de la variable binaria `reordered`.

In [18]:
print("Registros de pedidos:", orders.shape[0])
print("Productos en el catálogo:", products.shape[0])
print("Promedio de order_number:", orders["order_number"].mean())
print("Promedio de days_since_prior_order:", orders["days_since_prior_order"].mean())
print("Filas con reordered == 1:", order_products.loc[order_products["reordered"] == 1, "reordered"].count())
print("Suma de reordered:", order_products["reordered"].sum())

print("\nEstadísticas de order_hour_of_day:")
print(orders["order_hour_of_day"].describe())
print("\nEstadísticas de add_to_cart_order:")
print(order_products["add_to_cart_order"].describe())

Registros de pedidos: 478967
Productos en el catálogo: 49694
Promedio de order_number: 17.157808366755955
Promedio de days_since_prior_order: 11.101750979677794


Filas con reordered == 1: 2683838
Suma de reordered: 2683838

Estadísticas de order_hour_of_day:
count    478967.000000
mean         13.446676
std           4.224986
min           0.000000
25%          10.000000
50%          13.000000
75%          16.000000
max          23.000000
Name: order_hour_of_day, dtype: float64

Estadísticas de add_to_cart_order:


count    4.544171e+06
mean     8.351669e+00
std      7.080368e+00
min      1.000000e+00
25%      3.000000e+00
50%      6.000000e+00
75%      1.100000e+01
max      6.400000e+01
Name: add_to_cart_order, dtype: float64


In [19]:
print("IDs de pedido únicos:", orders["order_id"].nunique())
print("Usuarios únicos:", orders["user_id"].nunique())
print("Filas con order_id duplicado:", orders["order_id"].duplicated().sum())
print("IDs de producto únicos en el catálogo:", products["product_id"].nunique())

IDs de pedido únicos: 478952
Usuarios únicos: 157437
Filas con order_id duplicado: 15
IDs de producto únicos en el catálogo: 49694


### Interpretación de los indicadores

| Pregunta | Valor obtenido | Interpretación |
|---|---:|---|
| ¿Cuántos registros de pedidos contiene el archivo? | 478.967 filas; 478.952 `order_id` distintos | Hay 15 filas cuyo `order_id` está repetido. El número de filas no es exactamente el número de pedidos únicos. |
| ¿Cuántos registros tiene el catálogo de productos? | 49.694 filas y 49.694 `product_id` distintos | Cada identificador de producto aparece una vez en este catálogo. |
| ¿Cuál es el promedio de `order_number`? | 17,16 | Es el promedio de esa columna en las filas de `orders`; no es el promedio calculado sobre usuarios únicos. |
| ¿Cuál es el promedio de `days_since_prior_order`? | 11,10 días | Se calcula sobre los 450.148 valores disponibles; no incluye 28.819 valores ausentes. |
| ¿Cuántos artículos tienen `reordered == 1`? | 2.683.838 de 4.545.007 filas (59,05 %) | Son filas de artículos en pedidos marcadas como recompra; no clientes únicos. |
| ¿Cuál es la suma de `reordered`? | 2.683.838 | Como la columna contiene 0 y 1, su suma coincide con el número de filas marcadas con 1. |
| ¿Qué describe `order_hour_of_day`? | Media 13,45; mediana 13; mínimo 0; máximo 23 | Los registros cubren todas las horas de 0 a 23; la media está cerca de las 13:27. |
| ¿Qué describe `add_to_cart_order`? | 4.544.171 valores disponibles; media 8,35; mediana 6; mínimo 1; máximo 64 | La posición típica está alrededor de 6; hay 836 valores ausentes, excluidos de estas estadísticas. |

## Momento 6. Resolución del problema y conclusiones

### Respuestas a las tres preguntas iniciales

1. **Pedidos a las 10:00 y pedidos con `order_number > 10` dentro de ese grupo:** 40.578 registros cumplen la hora; 21.537 cumplen ambas condiciones. Se cuentan filas de `orders`; hay 15 filas con `order_id` repetido en el archivo.
2. **Filas de artículos con recompra:** 2.683.838 filas de `order_products` tienen `reordered == 1`, equivalentes al 59,05 % de las filas de esa tabla. Esto no representa el número de usuarios que recompraron.
3. **Promedio de días hasta el pedido anterior:** 11,10 días, calculado sobre 450.148 valores disponibles de `days_since_prior_order`; 28.819 filas no tienen ese dato.

### Síntesis

La exploración permitió identificar que `orders` describe pedidos y usuarios, `products` contiene el catálogo y `order_products` relaciona productos con pedidos. Las tablas `aisles` y `departments` aportan nombres para clasificar los productos. Los filtros y `loc[]` permitieron aislar registros que cumplen condiciones concretas; las operaciones `mean()`, `sum()`, `count()` y `describe()` resumieron columnas numéricas y de recompra.

### Limitaciones observadas

- Hay valores ausentes en `orders.days_since_prior_order`, `products.product_name` y `order_products.add_to_cart_order`. No se corrigieron ni eliminaron, tal como indica la actividad.
- `orders` contiene 15 filas con `order_id` duplicado; por tanto, el total de filas no equivale exactamente al número de pedidos distintos.
- `order_products` contiene 4.545.007 filas para 450.046 pedidos distintos, y 28.906 pedidos de `orders` no aparecen en esa tabla. Los conteos de filas de artículos no deben presentarse como pedidos ni como clientes.
- El archivo no incluye un diccionario que confirme qué día representa cada valor de `order_dow`; por eso el filtro se reporta como código 0.

### Tres conclusiones

1. En el archivo hay 40.578 registros de pedidos a las 10:00 y 21.537 que además tienen `order_number > 10`; el filtro combinado reduce el conjunto a los registros que cumplen ambos criterios.
2. La recompra aparece en 2.683.838 filas de artículos (59,05 %). Es una proporción de registros artículo-pedido y no permite concluir directamente qué porcentaje de clientes repite compras.
3. El intervalo medio entre pedidos es 11,10 días entre los valores informados. La presencia de 28.819 datos ausentes limita esta interpretación, por lo que no se imputaron ni se descartaron registros para esta exploración.

### Lista final de verificación basada en la guía

- [x] Se describen el problema, los cinco archivos, los identificadores, las relaciones y tres preguntas de análisis.
- [x] Se cargan los cinco CSV con `sep=';'` y se incluyen `head()`, `tail()` y `shape`.
- [x] Se revisan `columns`, `dtypes` e `info(show_counts=True)` para las cinco tablas.
- [x] Se documentan valores ausentes sin corregir ni eliminar filas.
- [x] Se incluyen los ocho ejercicios, los dos filtros simultáneos y las preguntas de interpretación.
- [x] Se calculan e interpretan los indicadores solicitados y se presentan tres conclusiones.
- [ ] Ejecutar todas las celdas en orden en Jupyter con el kernel de `.venv` y guardar el cuaderno con sus salidas.
- [ ] Comparar el producto final con la lista de chequeo oficial cuando se comparta; el documento recibido solo indica que existe, no contiene sus criterios.